# RSA Representation Methods Analysis - Wilson Dataset

This notebook tests multiple representation encoding methods for RSA analysis
to assess robustness of imagery vs perception findings.

## Analysis Pipeline
1. Load preprocessed epochs (imagery and perception conditions)
2. Extract representations using different encoding schemes
3. Compute RDMs for imagery and perception separately
4. Compare imagery vs perception RDMs using Spearman correlation
5. Test significance with permutation tests
6. Compare results across representation methods
7. Visualize RDMs and summary statistics

## Representation Methods
- **power_bands**: Average power in frequency bands across all channels
- **channels**: Mean activity per channel (spatial patterns)
- **channel_x_band**: Power per channel per band (full spatial-frequency matrix)
- **time_windows**: Representations from different temporal windows
- **erp_features**: Peak amplitude, mean amplitude, area under curve
- **time_frequency**: Time-frequency decomposition features

## References
- Anwar et al. (2024) - Representation method testing approach
- Wilson et al. - Original imagery vs perception study


In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import analysis modules
import sys
sys.path.append('..')
from feature_extraction.extract_representations import (
    extract_representation_vector,
    get_representation_methods
)
from analysis.rsa_representation_methods import (
    test_all_methods,
    plot_method_comparison,
    plot_rdm_comparison
)
from config.pipeline_config import REPRESENTATION_METHODS

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Configuration
PROCESSED_DIR = Path('../data/processed')
RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('../figures')
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
FIGURES_DIR.mkdir(exist_ok=True, parents=True)

print('Environment setup complete')
print(f'Available representation methods: {get_representation_methods()}')


## 1. Load Preprocessed Data

Load the preprocessed epochs for imagery and perception conditions.


In [ ]:
# Load preprocessed epochs
# TODO: Update with actual paths to Wilson dataset
# epochs_imagery = mne.read_epochs(PROCESSED_DIR / 'imagery_epochs-epo.fif')
# epochs_perception = mne.read_epochs(PROCESSED_DIR / 'perception_epochs-epo.fif')

# For demonstration, create sample epochs with imagery and perception conditions
# This will be replaced with actual data loading

# Create sample data structure
info = mne.create_info(
    ch_names=['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2'],
    sfreq=250,
    ch_types='eeg'
)

n_epochs_per_condition = 40
n_channels = 10
n_times = 300  # 1.2 seconds at 250 Hz

# Create imagery epochs
imagery_data = np.random.randn(n_epochs_per_condition, n_channels, n_times)
imagery_events = np.array([[i * 1000, 0, (i % 4) + 1] for i in range(n_epochs_per_condition)])
imagery_event_id = {'stim1': 1, 'stim2': 2, 'stim3': 3, 'stim4': 4}
epochs_imagery = mne.EpochsArray(imagery_data, info, imagery_events, 
                                 tmin=-0.2, event_id=imagery_event_id)

# Create perception epochs
perception_data = np.random.randn(n_epochs_per_condition, n_channels, n_times)
perception_events = np.array([[i * 1000, 0, (i % 4) + 1] for i in range(n_epochs_per_condition)])
perception_event_id = {'stim1': 1, 'stim2': 2, 'stim3': 3, 'stim4': 4}
epochs_perception = mne.EpochsArray(perception_data, info, perception_events,
                                    tmin=-0.2, event_id=perception_event_id)

print(f'Loaded imagery epochs: {len(epochs_imagery)} epochs, {len(epochs_imagery.ch_names)} channels')
print(f'Loaded perception epochs: {len(epochs_perception)} epochs, {len(epochs_perception.ch_names)} channels')
print(f'Conditions: {list(imagery_event_id.keys())}')
print(f'Time window: {epochs_imagery.times[0]:.3f} to {epochs_imagery.times[-1]:.3f} s')


## 2. Test All Representation Methods

Test imagery-perception similarity across all representation encoding methods.


In [ ]:
# Test all representation methods
# This will compute RDMs for imagery and perception separately,
# then compare them using Spearman correlation with permutation testing

print('Testing all representation methods...')
print('='*60)

results = test_all_methods(
    epochs_imagery,
    epochs_perception,
    methods=None,  # None = test all methods
    n_permutations=1000,  # Use 1000 for faster demo; use 10000 for final analysis
    metric='correlation'
)

print('\n' + '='*60)
print('Analysis complete!')


## 3. Summary Statistics

Create summary table of results across all methods.


In [ ]:
# Create summary DataFrame
summary_data = []

for method, result in results.items():
    if 'error' not in result:
        summary_data.append({
            'Method': method,
            'Correlation': result['correlation'],
            'P-value (correlation)': result['p_value_correlation'],
            'P-value (permutation)': result['p_value_permutation'],
            'Significant': 'Yes' if result['p_value_permutation'] < 0.05 else 'No'
        })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Correlation', ascending=False)

print('Summary of Results:')
print('='*60)
print(summary_df.to_string(index=False))
print('='*60)

# Save summary
summary_df.to_csv(RESULTS_DIR / 'representation_methods_summary.csv', index=False)
print(f'\nSaved summary to {RESULTS_DIR / "representation_methods_summary.csv"}')
